In [2]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import os
from sklearn.metrics import roc_auc_score
from dataloader import dataLoader, load_data_with_outliers, extract_resnet50_feature
from method import DaDTAnomalyDetector
import cv2
import time
import os
import asyncio

In [3]:
setIndex = 1
featureType = 'ResNet' # 'ResNet', 'Clip'
all_feats, all_labels, dataset_name = dataLoader(setIndex, featureType)
print('dataset: ', dataset_name)

feature type:  ResNet
dataset:  CIFAR-10


In [4]:
target_class = 0
p_anon = 0.3

data, gt = load_data_with_outliers(all_feats, all_labels, target_class, p_anon)

In [5]:
clf = DaDTAnomalyDetector()

score = clf.dadt_(data)
auc = roc_auc_score(gt, score)
print('auc: ', format(roc_auc_score(gt, score), '.3f'))

auc:  0.945


In [6]:
#Test runs for a moving window style solution
new_feat = extract_resnet50_feature("test1.jpg")
data_with_new = np.vstack([data, new_feat])

scores, labels = clf.predict_labels(data_with_new)
print(f"Mean Score = {np.mean(scores)}")
print(f"Percent Outliers = {np.mean(labels)}")
inference = labels[-1] 
print(f"Score: {scores[-1]}")
if inference == 1:
    print("OUTLIER")
else:
    print("INLIER")

Threshold: 0.08487182855606079
Mean Score = 0.05107975751161575
Percent Outliers = 0.32722818478768084
Score: 0.08749431371688843
OUTLIER


In [ ]:
#Get a list of features from a stream and collect some data
stream = "http://158.58.130.148/mjpg/video.mjpg" # This is for some random hotel camera, idk we can get something else as well
os.makedirs("StreamData", exist_ok=True)

cap = cv2.VideoCapture(stream)
features = []
for i in range(0,100):
    # Grab a frame and get features
    ret, frame = cap.read()
    
    timestamp = int(time.time())
    filepath = os.path.join("StreamData", f"snapshot_{timestamp}.jpg")
    
    try:
        cv2.imwrite(filepath, frame)

    except Exception as e:
        print(f"Failed to get frame: {e}")
        continue

    print(f"Saved {filepath}")

    #store feature
    feat = extract_resnet50_feature(filepath)
    features.append(feat)

    # await sleep

np.save("features.npy", np.array(features))

Saved StreamData\snapshot_1764120424.jpg
Saved StreamData\snapshot_1764120429.jpg
Saved StreamData\snapshot_1764120430.jpg
Saved StreamData\snapshot_1764120435.jpg
Saved StreamData\snapshot_1764120439.jpg
Saved StreamData\snapshot_1764120440.jpg
Saved StreamData\snapshot_1764120451.jpg
Saved StreamData\snapshot_1764120456.jpg
Saved StreamData\snapshot_1764120460.jpg
Saved StreamData\snapshot_1764120465.jpg
Saved StreamData\snapshot_1764120475.jpg
Saved StreamData\snapshot_1764120482.jpg
Saved StreamData\snapshot_1764120484.jpg
Saved StreamData\snapshot_1764120489.jpg
Saved StreamData\snapshot_1764120494.jpg
Saved StreamData\snapshot_1764120496.jpg
Saved StreamData\snapshot_1764120499.jpg
Saved StreamData\snapshot_1764120512.jpg
Saved StreamData\snapshot_1764120515.jpg
Saved StreamData\snapshot_1764120524.jpg
Saved StreamData\snapshot_1764120531.jpg
Saved StreamData\snapshot_1764120538.jpg
Saved StreamData\snapshot_1764120543.jpg
Saved StreamData\snapshot_1764120553.jpg
Saved StreamData

In [ ]:
#play with new features
os.chdir("C:\\Users\\joshm\\repo\\FlexUOD-Extension")
all_feats = np.load("features.npy")



#idk why it isnt working and doesnt really matter 
#but could you this for testing your own datasets from a video:
# features = []

# for file in os.listdir():
#     feat = extract_resnet50_feature(filepath)
#     features.append(feat)


clf_test = DaDTAnomalyDetector()

score_test, labels_test = clf_test.predict_labels(all_feats)

print(f"Mean Score = {np.mean(score_test)}")
print(f"Percent Outliers = {np.mean(labels_test)}")


Threshold: 0.049345433712005615
Mean Score = 0.04179222881793976
Percent Outliers = 0.0
